## RL POSTERIOR RESULTS


In [92]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("final_dataset.csv")

# -----------------------------------
# BASE CLEAN
# -----------------------------------
df = df_raw.copy()

# Remove practice trials if column exists
if "practice_trial" in df.columns:
    df = df[df["practice_trial"] == "real"]

# Extract numeric part of subject identifiers
df["subject_id"] = df["subject_id"].astype(str).str.extract("(\d+)").astype(int)


# Choice recoding: convert {0,1} to {1,2}
df["choice_1"] = df["choice_1"].astype(int).replace({0: 1, 1: 2})
df["choice_2"] = df["choice_2"].astype(int).replace({0: 1, 1: 2})

# -----------------------
# FIX STATE LABELS
# original states: {1,2}
# RLDDM expects {2,3}
# -----------------------
df["state"] = df["state"].astype(int).replace({
    0: 2,   # if any exist
    1: 2,
    2: 3,
    3: 3     # safety in case
})

print("\nUnique state values after mapping:", df["state"].unique())

# -----------------------
# Clean RT2
df["rt_2"] = pd.to_numeric(df["rt_2"], errors="coerce")

# Convert milliseconds to seconds if >10
df["rt_2"] = np.where(df["rt_2"] > 10, df["rt_2"] / 1000, df["rt_2"])

# Floor RT2 at a realistic minimum (150 ms)
df["rt_2"] = df["rt_2"].fillna(0.15)
df["rt_2"] = df["rt_2"].clip(lower=0.15)


df_clean = df.copy()

print("\nFinal df_clean shape:", df_clean.shape)
print(df_clean.head())



Unique state values after mapping: [3 2]

Final df_clean shape: (30200, 13)
   subject_id practice_trial  trial transition  reward  choice_1  choice_2  \
0           1           real      1     common       1         2         2   
1           1           real      2     common       0         2         2   
2           1           real      3       rare       0         2         2   
3           1           real      4     common       1         2         2   
4           1           real      5       rare       1         2         2   

       rt_1      rt_2  state  gender        age    age_group  
0  1131.480  1.457620      3  Female  17.055556  Adolescents  
1   639.000  0.483590      3  Female  17.055556  Adolescents  
2   264.190  0.718195      3  Female  17.055556  Adolescents  
3   302.265  1.507225      3  Female  17.055556  Adolescents  
4   321.685  1.751195      3  Female  17.055556  Adolescents  


<>:16: SyntaxWarning: invalid escape sequence '\d'
<>:16: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Taychaz\AppData\Local\Temp\ipykernel_14428\2269731119.py:16: SyntaxWarning: invalid escape sequence '\d'
  df["subject_id"] = df["subject_id"].astype(str).str.extract("(\d+)").astype(int)


In [93]:
def build_data_rlddm(df):
    df = df.sort_values(["subject_id", "trial"]).reset_index(drop=True)

    subjects = df["subject_id"].unique()
    S = len(subjects)

    T = df.groupby("subject_id").size().values
    T_max = T.max()

    c1 = np.zeros((S, T_max), dtype=int)
    c2 = np.zeros((S, T_max), dtype=int)
    r  = np.zeros((S, T_max))
    s2 = np.zeros((S, T_max), dtype=int)
    rt2 = np.zeros((S, T_max))

    rt2_min = np.zeros(S)

    for i, sid in enumerate(subjects):
        tmp = df[df.subject_id == sid]
        T_s = len(tmp)

        c1[i,:T_s] = tmp["choice_1"].values
        c2[i,:T_s] = tmp["choice_2"].values
        r[i,:T_s]  = tmp["reward"].values
        s2[i,:T_s] = tmp["state"].values
        rt2[i,:T_s] = tmp["rt_2"].values

        # subject minimum RT
        rt2_min[i] = max(0.15, tmp["rt_2"].min())


    # Prior first choice per subject
    prior_choice = np.zeros(S)
    for i, sid in enumerate(subjects):
        prior_choice[i] = df[df.subject_id == sid]["choice_1"].iloc[0]

    return {
        "S": S,
        "T": T.astype(int),
        "T_max": int(T_max),
        "c1": c1,
        "c2": c2,
        "r": r,
        "s2raw": s2,
        "rt2": rt2,
        "rt2_min": rt2_min,
        "ter_eps": 0.01,
        "prior_choice": prior_choice.astype(int),
        "t_common": 0.7
    }


In [100]:
data_rlddm = build_data_rlddm(df_clean)

print("S =", data_rlddm["S"])
print("T_max =", data_rlddm["T_max"])
print("State min/max =", data_rlddm["s2raw"].min(), data_rlddm["s2raw"].max())

print("RT2 global min =", data_rlddm["rt2"].min())


S = 151
T_max = 200
State min/max = 2 3
RT2 global min = 0.15


In [109]:
from cmdstanpy import CmdStanModel

model_rlddm = CmdStanModel(stan_file="hybrid_rl_ddm.stan")

fit_rlddm_test = model_rlddm.sample(
    data=data_rlddm,
    chains=1,
    parallel_chains=1,
    iter_warmup=100,
    iter_sampling=100,
    adapt_delta=0.9,
    seed=123,
    show_console=True
)



16:30:26 - cmdstanpy - INFO - Chain [1] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 100
Chain [1] num_warmup = 100
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.9
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = C:\Users\Taychaz\AppData\Local\Temp\tmprr020z0a\oii8ctr_.json
Chain [1] init = 2 (Default)
Chain [1] random
Chain [1] seed = 123
Chain [1]

16:34:58 - cmdstanpy - INFO - Chain [1] done processing
16:34:58 - cmdstanpy - ERROR - Chain [1] error: code '3221225477' 


Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 

RuntimeError: Error during sampling:

Command and output files:
RunSet: chains=1, chain_ids=[1], num_processes=1
 cmd (chain 1):
	['C:\\Users\\Taychaz\\Desktop\\Oguz\\Thesis\\thesis-project\\data\\hybrid_rl_ddm.exe', 'id=1', 'random', 'seed=123', 'data', 'file=C:\\Users\\Taychaz\\AppData\\Local\\Temp\\tmprr020z0a\\oii8ctr_.json', 'output', 'file=C:\\Users\\Taychaz\\AppData\\Local\\Temp\\tmprr020z0a\\hybrid_rl_ddmiafa1klv\\hybrid_rl_ddm-20251204163026.csv', 'method=sample', 'num_samples=100', 'num_warmup=100', 'algorithm=hmc', 'adapt', 'engaged=1', 'delta=0.9']
 retcodes=[3221225477]
 per-chain output files (showing chain 1 only):
 csv_file:
	C:\Users\Taychaz\AppData\Local\Temp\tmprr020z0a\hybrid_rl_ddmiafa1klv\hybrid_rl_ddm-20251204163026.csv
 console_msgs (if any):
	C:\Users\Taychaz\AppData\Local\Temp\tmprr020z0a\hybrid_rl_ddmiafa1klv\hybrid_rl_ddm-20251204163026_0-stdout.txt

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from cmdstanpy import CmdStanModel

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("final_dataset.csv")

# Standardize column names
df = df.rename(columns={
    "subject_id": "subject",
    "choice_1": "c1",
    "choice_2": "c2",
    "state": "state",
    "reward": "reward",
    "rt_2": "rt2"
})

# ============================================================
# 2. CLEAN DATA (scientific, strict)
# ============================================================

# Convert to numeric
df["c1"] = pd.to_numeric(df["c1"], errors="coerce")
df["c2"] = pd.to_numeric(df["c2"], errors="coerce")
df["state"] = pd.to_numeric(df["state"], errors="coerce")
df["reward"] = pd.to_numeric(df["reward"], errors="coerce")
df["rt2"] = pd.to_numeric(df["rt2"], errors="coerce")

# Filter valid trials
df = df[
    df["c1"].isin([1,2]) &
    df["c2"].isin([1,2]) &
    df["state"].isin([2,3]) &
    df["reward"].notna() &
    df["rt2"].notna() &
    (df["rt2"] > 0)
].copy()

df = df.sort_values(["subject", "trial"]).reset_index(drop=True)

# ============================================================
# 3. SUBJECT INDEXING
# ============================================================

def extract_number(s):
    # converts "sub103" -> 103
    return int("".join(filter(str.isdigit, s)))

subject_order = sorted(df["subject"].unique(), key=extract_number)
df["sub_index"] = df["subject"].apply(lambda x: subject_order.index(x) + 1)

S = df["sub_index"].nunique()
print("Subjects:", S)

# ============================================================
# 4. BUILD MATRICES FOR STAN
# ============================================================

T_list = df.groupby("sub_index").size().values.astype(int)
T_max = T_list.max()
print("Max trials:", T_max)

# Initialize matrices
c1 = np.ones((S, T_max), dtype=int)
c2 = np.ones((S, T_max), dtype=int)
r = np.zeros((S, T_max), dtype=float)
s2raw = np.ones((S, T_max), dtype=int)
rt = np.zeros((S, T_max), dtype=float)

prior_choice = np.ones(S, dtype=int)

# Fill matrices
for s in range(1, S+1):
    sub_data = df[df["sub_index"] == s].sort_values("trial")
    T_s = len(sub_data)

    c1[s-1, :T_s] = sub_data["c1"].values
    c2[s-1, :T_s] = sub_data["c2"].values
    r[s-1, :T_s] = sub_data["reward"].values
    s2raw[s-1, :T_s] = sub_data["state"].values
    rt[s-1, :T_s] = sub_data["rt2"].values

    prior_choice[s-1] = int(sub_data.iloc[0]["c1"])

# ============================================================
# 5. FINAL SAFETY CHECK (guarantees no zeros)
# ============================================================

c1[c1 < 1] = 1
c2[c2 < 1] = 1
s2raw[s2raw < 1] = 2

rt_floor = 0.05
rt[rt <= 0] = rt_floor

# ============================================================
# 6. BUILD STAN DATA DICTS
# ============================================================

data_rl = {
    "S": S,
    "T_max": int(T_max),
    "T": T_list.tolist(),
    "c1": c1.tolist(),
    "c2": c2.tolist(),
    "r": r.tolist(),
    "s2raw": s2raw.tolist(),
    "prior_choice": prior_choice.tolist(),
    "t_common": 0.7,
}

rt2_min = np.array([rt[i, :T_list[i]].min() for i in range(S)])

data_rlddm = {
    "S": S,
    "T_max": int(T_max),
    "T": T_list.tolist(),
    "c1": c1.tolist(),
    "c2": c2.tolist(),
    "r": r.tolist(),
    "s2raw": s2raw.tolist(),
    "rt2": rt.tolist(),
    "rt2_min": rt2_min.tolist(),
    "rt2_min_global": float(rt2_min.min()),
    "ter_eps": 0.001,
    "prior_choice": prior_choice.tolist(),
    "t_common": 0.7,
}

print("RL and RLDDM data prepared.")

# ============================================================
# 7. RUN STAN MODELS
# ============================================================

model_rl = CmdStanModel(stan_file="stan_hybrid_rlm.stan")
model_rlddm = CmdStanModel(stan_file="hybrid_rl_ddm.stan")

# fit_rl = model_rl.sample(
#     data=data_rl,
#     chains=4,
#     parallel_chains=4,
#     iter_warmup=1000,
#     iter_sampling=1000,
#     seed=123
# )

# print("RL model finished.")

# # RLDDM is slower
# fit_rlddm = model_rlddm.sample(
#     data=data_rlddm,
#     chains=4,
#     parallel_chains=4,
#     iter_warmup=1000,
#     iter_sampling=1000,
#     seed=123,
#     show_console=True
# )

# print("RLDDM model finished.")


Subjects: 151
Max trials: 200
RL and RLDDM data prepared.


13:36:15 - cmdstanpy - INFO - Chain [1] start processing
13:36:15 - cmdstanpy - INFO - Chain [2] start processing
13:36:15 - cmdstanpy - INFO - Chain [3] start processing
13:36:15 - cmdstanpy - INFO - Chain [4] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 1000 (Default)
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.8 (Default)
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = C:\Users\Taychaz\AppData\Local\Temp\tmprr020z0a\47sq9qja.json
Chain [1] init = 2 (Default)
Chain [1] rando

13:36:16 - cmdstanpy - INFO - Chain [3] done processing
13:36:16 - cmdstanpy - ERROR - Chain [3] error: code '1' Operation not permitted
13:36:16 - cmdstanpy - INFO - Chain [4] done processing
13:36:16 - cmdstanpy - ERROR - Chain [4] error: code '1' Operation not permitted
13:36:16 - cmdstanpy - INFO - Chain [2] done processing
13:36:16 - cmdstanpy - ERROR - Chain [2] error: code '1' Operation not permitted
13:36:16 - cmdstanpy - INFO - Chain [1] done processing
13:36:16 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted


Chain [1] Rejecting initial value:
Chain [1] Log probability evaluates to log(0), i.e. negative infinity.
Chain [1] Stan can't start sampling from this initial value.
Chain [2] Rejecting initial value:
Chain [2] Log probability evaluates to log(0), i.e. negative infinity.
Chain [2] Stan can't start sampling from this initial value.
Chain [3] Rejecting initial value:
Chain [3] Log probability evaluates to log(0), i.e. negative infinity.
Chain [3] Stan can't start sampling from this initial value.
Chain [4] Rejecting initial value:
Chain [4] Log probability evaluates to log(0), i.e. negative infinity.
Chain [4] Stan can't start sampling from this initial value.
Chain [1] Rejecting initial value:
Chain [1] Log probability evaluates to log(0), i.e. negative infinity.
Chain [1] Stan can't start sampling from this initial value.
Chain [2] Rejecting initial value:
Chain [2] Log probability evaluates to log(0), i.e. negative infinity.
Chain [2] Stan can't start sampling from this initial value

RuntimeError: Error during sampling:

Command and output files:
RunSet: chains=4, chain_ids=[1, 2, 3, 4], num_processes=4
 cmd (chain 1):
	['C:\\Users\\Taychaz\\Desktop\\Oguz\\Thesis\\thesis-project\\data\\hybrid_rl_ddm.exe', 'id=1', 'random', 'seed=123', 'data', 'file=C:\\Users\\Taychaz\\AppData\\Local\\Temp\\tmprr020z0a\\47sq9qja.json', 'output', 'file=C:\\Users\\Taychaz\\AppData\\Local\\Temp\\tmprr020z0a\\hybrid_rl_ddm8fz8nd8f\\hybrid_rl_ddm-20251204133615_1.csv', 'method=sample', 'num_samples=1000', 'num_warmup=1000', 'algorithm=hmc', 'adapt', 'engaged=1']
 retcodes=[1, 1, 1, 1]
 per-chain output files (showing chain 1 only):
 csv_file:
	C:\Users\Taychaz\AppData\Local\Temp\tmprr020z0a\hybrid_rl_ddm8fz8nd8f\hybrid_rl_ddm-20251204133615_1.csv
 console_msgs (if any):
	C:\Users\Taychaz\AppData\Local\Temp\tmprr020z0a\hybrid_rl_ddm8fz8nd8f\hybrid_rl_ddm-20251204133615_0-stdout.txt

## Posterior Predictive Accuracy for RL

In [8]:
import numpy as np
import pandas as pd

df_post = pd.read_csv("results/rl/posterior_samples_rl.csv")

# Extract posterior predictive draws
y1_cols = [c for c in df_post.columns if c.startswith("y1_rep")]
y1_rep = df_post[y1_cols].to_numpy()

# Actual data
S = data_rl["S"]
T = data_rl["T"]     # list of lengths per subject
c1 = np.array(data_rl["c1"])  # S x T_max

pp_acc_subject = []

offset = 0
for s in range(S):
    T_s = T[s]  # real trials for this subject
    
    # predicted values for only real trials
    pred = y1_rep[:, offset:offset+T_s]    # draws x T_s
    pred_mean = pred.mean(axis=0).round()   # convert to 1/2 choices
    
    real = c1[s, :T_s]                      # true choices
    
    acc = (pred_mean == real).mean()
    pp_acc_subject.append(acc)
    
    offset += T_s

print("Mean predictive accuracy:", np.mean(pp_acc_subject))
print("Median predictive accuracy:", np.median(pp_acc_subject))


Mean predictive accuracy: 0.5043858355566369
Median predictive accuracy: 0.5


In [9]:
import numpy as np
import pandas as pd

df_post = pd.read_csv("results/rl/posterior_samples_rl.csv")

# Extract posterior predictive draws
y1_cols = [c for c in df_post.columns if c.startswith("y1_rep")]
y2_cols = [c for c in df_post.columns if c.startswith("y2_rep")]

y1_rep = df_post[y1_cols].to_numpy()   # draws x (S*T_max)
y2_rep = df_post[y2_cols].to_numpy()

S = data_rl["S"]
T = data_rl["T"]        # real trials per subject
c1 = np.array(data_rl["c1"])
c2 = np.array(data_rl["c2"])

# Accuracy containers
acc_stage1 = []
acc_stage2 = []

offset = 0
for s in range(S):
    T_s = T[s]

    # Stage 1
    pred1 = y1_rep[:, offset:offset+T_s].mean(axis=0).round()
    real1 = c1[s, :T_s]
    acc_stage1.append((pred1 == real1).mean())

    # Stage 2
    pred2 = y2_rep[:, offset:offset+T_s].mean(axis=0).round()
    real2 = c2[s, :T_s]
    acc_stage2.append((pred2 == real2).mean())

    offset += T_s

print("Stage 1 predictive accuracy:", np.mean(acc_stage1))
print("Stage 2 predictive accuracy:", np.mean(acc_stage2))


Stage 1 predictive accuracy: 0.5043858355566369
Stage 2 predictive accuracy: 0.5131860814324858


In [12]:
# --- REBUILD the exact model df ---
def preprocess_for_model(df_raw):
    df = df_raw.copy()

    df = df[(df["choice_1"].isin([1,2])) & (df["choice_2"].isin([1,2]))]
    df = df[df["state"].isin([2,3])]
    df = df[df["reward"].notna()]

    df["rt_2"] = pd.to_numeric(df["rt_2"], errors='coerce')
    df = df[df["rt_2"] > 0]

    df = df.sort_values(["subject_id", "trial"]).reset_index(drop=True)
    return df

df_model = preprocess_for_model(df_raw)
df = df_model.copy()  # <-- THIS is the good df

# ---- actual stay indicator ----
df["stay"] = df.groupby("subject_id")["choice_1"].diff().eq(0).astype(int)

# ---- posterior samples ----
df_post = pd.read_csv("results/rl/posterior_samples_rl.csv")
y1_cols = [c for c in df_post.columns if c.startswith("y1_rep")]
y1_rep = df_post[y1_cols].to_numpy()

S = data_rl["S"]
T = data_rl["T"]

# ---- posterior predictive choices ----
pp_choice1 = []
offset = 0
for s in range(S):
    T_s = T[s]
    pred = y1_rep[:, offset:offset+T_s].mean(axis=0).round()
    pp_choice1.extend(pred)
    offset += T_s

df["pp_choice1"] = pp_choice1  # now works!

# ---- posterior predictive stay ----
df["pp_stay"] = df.groupby("subject_id")["pp_choice1"].diff().eq(0).astype(int)

# ---- grouped results ----
pp_stay = df.groupby(["transition", "reward"])["pp_stay"].mean()
print(pp_stay)


transition  reward
common      0         0.495202
            1         0.501113
rare        0         0.491570
            1         0.487555
Name: pp_stay, dtype: float64


In [13]:
import numpy as np

rmse_stage1 = []
rmse_stage2 = []

offset = 0
for s in range(S):
    T_s = T[s]

    pred1 = y1_rep[:, offset:offset+T_s].mean(axis=0)
    pred2 = y2_rep[:, offset:offset+T_s].mean(axis=0)

    real1 = c1[s, :T_s]
    real2 = c2[s, :T_s]

    rmse_stage1.append(np.sqrt(np.mean((pred1 - real1)**2)))
    rmse_stage2.append(np.sqrt(np.mean((pred2 - real2)**2)))

    offset += T_s

print("RMSE Stage 1:", np.mean(rmse_stage1))
print("RMSE Stage 2:", np.mean(rmse_stage2))


RMSE Stage 1: 0.5872005253852313
RMSE Stage 2: 0.5600464129270281


In [15]:
import numpy as np

# Map subject_id → age_group
df["subject"] = df["subject_id"]

sub_to_age = (
    df.drop_duplicates("subject")[["subject", "age_group"]]
      .set_index("subject")
      .to_dict()["age_group"]
)

# Storage for accuracies by age group
age_acc1 = {"Children": [], "Adolescents": [], "Adults": []}
age_acc2 = {"Children": [], "Adolescents": [], "Adults": []}

subjects = list(sub_to_age.keys())      # list of subject IDs
S = data_rl["S"]
T = data_rl["T"]

offset = 0  # offsets inside posterior predictive vector (flat)

for i in range(S):

    subject = subjects[i]
    age = sub_to_age[subject]
    T_s = T[i]

    # posterior predictive (draws × trials)
    pred1 = y1_rep[:, offset:offset+T_s].mean(axis=0).round()
    pred2 = y2_rep[:, offset:offset+T_s].mean(axis=0).round()

    # true behavior
    real1 = np.array(data_rl["c1"][i])[:T_s]
    real2 = np.array(data_rl["c2"][i])[:T_s]

    # accuracy per subject
    acc1 = (pred1 == real1).mean()
    acc2 = (pred2 == real2).mean()

    age_acc1[age].append(acc1)
    age_acc2[age].append(acc2)

    offset += T_s


print("Stage 1 accuracy by age:", {k: np.mean(v) for k,v in age_acc1.items()})
print("Stage 2 accuracy by age:", {k: np.mean(v) for k,v in age_acc2.items()})


Stage 1 accuracy by age: {'Children': np.float64(0.5128909922409771), 'Adolescents': np.float64(0.5039433936256128), 'Adults': np.float64(0.49648121324946354)}
Stage 2 accuracy by age: {'Children': np.float64(0.5108287436744884), 'Adolescents': np.float64(0.5121938445241644), 'Adults': np.float64(0.5164699781641707)}
